In [ ]:
import sys, os

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
import pandas as pd
import numpy as np
import torch
from fixedincomelib import *

print("Fixed Income Library is loaded.")

Fixed Income Library is loaded.


### Create Build Method Collection (USD)

Curves:
- `SOFR-1B` -- root overnight index curve, calibrated off 10 tenor points
- `SOFR-1B-FLAT` -- flat (zero-spread) funding curve over `SOFR-1B`, calibrated off a single tenor point
- `USD` -- common build method tying the currency to its CSA/funding curve

Both components are calibrated directly from `INSTANTANEOUS FORWARD RATE` state data.

In [ ]:
bm_list = []

bm_list.append(
    qfCreateBuildMethod(
        "YC_OVERNIGHT_INDEX_ELEMENT",
        {
            "TARGET": "SOFR-1B",
            "INSTANTANEOUS FORWARD RATE": "USD-SOFR-OIS-1B-IFR",
        },
    )
)

bm_list.append(
    qfCreateBuildMethod(
        "YC_FUNDING_ELEMENT",
        {
            "TARGET": "SOFR-1B-FLAT",
            "REFERENCE": "SOFR-1B",
            "INSTANTANEOUS FORWARD RATE": "USD-SOFR-OIS-1B-FLAT-IFR",
        },
    )
)

bm_list.append(
    qfCreateBuildMethod(
        "YC_COMMON",
        {
            "TARGET": "USD",
            "FUNDING PARAMETERS": "SOFR-1B-FLAT",
            "SOLVER METHOD": "BRENT",
        },
    )
)

build_method_collection = qfCreateModelBuildMethodCollection(bm_list)
qfDisplayModelBuildMethodCollection(build_method_collection)

,Name,Value
0,YC_OVERNIGHT_INDEX_ELEMENT,SOFR-1B
1,YC_FUNDING_ELEMENT,SOFR-1B-FLAT
2,YC_COMMON,USD


### Create Data Collection (USD)

Synthetic instantaneous forward rate curve for `SOFR-1B` across 10 tenor points, and a single flat
zero-spread tenor point for `SOFR-1B-FLAT` (matching how `-FLAT` curves are used elsewhere as the
unspread CSA/funding overlay).

In [ ]:
data_type = "INSTANTANEOUS FORWARD RATE"
sofr_tenors = ["1M", "3M", "6M", "1Y", "2Y", "3Y", "5Y", "7Y", "10Y", "30Y"]
data_list = []

df = pd.DataFrame(index=sofr_tenors)
df["values"] = [0.0440, 0.0430, 0.0420, 0.0400, 0.0380, 0.0370, 0.0360, 0.0365, 0.0375, 0.0400]
data_list.append(qfCreateData1D(data_type, "USD-SOFR-OIS-1B-IFR", df))

df = pd.DataFrame(index=["10Y"])
df["values"] = [0.0]
data_list.append(qfCreateData1D(data_type, "USD-SOFR-OIS-1B-FLAT-IFR", df))

data_collection = qfCreateDataCollection(data_list)
data_collection.display()

,Data Shape,Data Type,Data Convention
0,DATA 1D,INSTANTANEOUS FORWARD RATE,USD-SOFR-OIS-1B-IFR
1,DATA 1D,INSTANTANEOUS FORWARD RATE,USD-SOFR-OIS-1B-FLAT-IFR


### Create Model Yield Curve (USD)

In [ ]:
value_date = "2026-06-25"
yc_usd = qfCreateModel(value_date, "YIELD_CURVE", data_collection, build_method_collection)
qfDisplayModelValueDate(yc_usd), qfDisplayModelType(yc_usd)

('2026-06-25', 'YIELD_CURVE')

### Inspect Resulting Model

In [ ]:
display(qfGetBuildMethodCollection(yc_usd).display())
display(qfGetDataCollectionFromModel(yc_usd).display())
print(f"USD yield curve model built successfully with {yc_usd.num_components} components.")

expiry_date = "2027-06-25"
print("SOFR-1B       DF:", qfDiscountFactor(yc_usd, "SOFR-1B", expiry_date))
print("SOFR-1B-FLAT  DF:", qfDiscountFactor(yc_usd, "SOFR-1B-FLAT", expiry_date))

,Name,Value
0,YC_OVERNIGHT_INDEX_ELEMENT,SOFR-1B
1,YC_FUNDING_ELEMENT,SOFR-1B-FLAT
2,YC_COMMON,USD


,Data Shape,Data Type,Data Convention
0,DATA 1D,INSTANTANEOUS FORWARD RATE,USD-SOFR-OIS-1B-IFR
1,DATA 1D,INSTANTANEOUS FORWARD RATE,USD-SOFR-OIS-1B-FLAT-IFR


USD yield curve model built successfully with 2 components.
SOFR-1B       DF: 0.9594847051399941
SOFR-1B-FLAT  DF: 0.9594847051399941


### Calibration Instruments and Risk (at par, against this curve)

`SOFR-1B` was calibrated directly from synthetic `INSTANTANEOUS FORWARD RATE` state data above, not
from real market instruments. In practice the two calibration instruments would be:
- `USD-SOFR-OIS` (an `OVERNIGHT INDEX SWAP` data convention) -- one par swap per `SOFR-1B` tenor
- `SOFR-1B-OVER-SOFR-1B-FLAT-ISZR` (an `IBOR SPREAD ZERO RATE` data convention, basis=`SOFR-1B`,
  reference=`SOFR-1B-FLAT`) -- the zero-spread instrument for `SOFR-1B-FLAT`, quoted at `spread=0`
  since `SOFR-1B-FLAT` was built flat over `SOFR-1B` above

This section builds those 11 instruments as real `Product` objects (10 par swaps + 1 zero-spread
product) and prices/risks them **against the curve already built above** -- it does not re-bootstrap
the curve from these instruments. `YieldCurveBuilder.create_model_yield_curve`
(`yield_curve/model_factory.py`) only supports the direct state-data path today (anything else
raises `NotImplementedError`), and `grad_at_par()` (which would be needed to build a real
calibration Jacobian / bucketed "risk at par" report, see the commented-out
`YieldCurve.calculate_model_jacobian`) is not implemented on any engine -- neither of those is
needed for what's done here. Each instrument's own `get_risk()` (already implemented, torch-autograd
based) is enough to get its curve-state sensitivity.

In [ ]:
from fixedincomelib.valuation.valuation_parameters import (
    ValuationParametersCollection,
    FundingIndexParameter,
    AnalyticValParam,
)
from fixedincomelib.valuation.valuation_engine import ValuationRequest
from fixedincomelib.valuation.valuation_engine_registry import ValuationEngineProductRegistry
import fixedincomelib.yield_curve.valuation_engine  # noqa: F401 -- import for its registry side effects

vpc = ValuationParametersCollection(
    [
        FundingIndexParameter(
            {
                "FUNDING INDEX": "SOFR-1B-FLAT",
                "CURRENCIES": "",
                "FUNDING INDICES": "",
                "UNDERLYING FUNDING INDEX": "",
            }
        ),
        AnalyticValParam({}),
    ]
)

#### 10 par `USD-SOFR-OIS` swaps, one per `SOFR-1B` tenor

Two passes per tenor: build a throwaway swap at `fixed_rate=0.` to read the curve-implied par rate
off `par_rate_or_spread()`, then rebuild at that rate so the swap actually held is a genuine par
instrument (PV ~ 0), the same way a real calibration instrument would be quoted at the market's par
rate.

In [ ]:
swap_engines = []
for tenor in sofr_tenors:
    guess = qfCreateProductFromDataConvention(value_date, "USD-SOFR-OIS", tenor, 0.0)
    guess_engine = ValuationEngineProductRegistry().new_valuation_engine(
        yc_usd, guess, vpc, ValuationRequest.PV
    )
    guess_engine.calculate_value()
    par_rate = guess_engine.par_rate_or_spread()

    par_swap = qfCreateProductFromDataConvention(value_date, "USD-SOFR-OIS", tenor, par_rate)
    engine = ValuationEngineProductRegistry().new_valuation_engine(
        yc_usd, par_swap, vpc, ValuationRequest.PV
    )
    engine.calculate_value()
    swap_engines.append((tenor, par_rate, engine))
    print(f"{tenor:>4s}  par rate={par_rate:.6f}  PV={float(engine.value): .2e}")

/var/folders/tn/lkm9_czj4ydgbt0b0nzzx2vm0000gn/T/ipykernel_11175/3208308497.py:12: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print(f'{tenor:>4s}  par rate={par_rate:.6f}  PV={float(engine.value): .2e}')


  1M  par rate=0.043410  PV= 0.00e+00
  3M  par rate=0.042902  PV= 0.00e+00
  6M  par rate=0.042491  PV= 0.00e+00
  1Y  par rate=0.041580  PV=-5.82e-11
  2Y  par rate=0.039892  PV= 0.00e+00
  3Y  par rate=0.039000  PV= 0.00e+00
  5Y  par rate=0.037924  PV= 2.33e-10
  7Y  par rate=0.037585  PV= 4.66e-10
 10Y  par rate=0.037611  PV= 4.66e-10
 30Y  par rate=0.039000  PV=-3.73e-09


#### Zero-spread calibration instrument for `SOFR-1B-FLAT`, at `spread=0`

`SOFR-1B-OVER-SOFR-1B-FLAT-ISZR` is an `IBOR SPREAD ZERO RATE` data convention
(`basis ibor index: SOFR-1B`, `reference index: SOFR-1B-FLAT`) -- dispatches to
`ProductFactory.create_zero_spread_product` -> `ProductIBORZeroSpread`, priced by the
`ValuationEngineProductZeroSpread` engine. `spread=0` matches how `SOFR-1B-FLAT` was itself built
(flat, zero-spread, over `SOFR-1B`) above, so PV should come out exactly `0`.

In [ ]:
zs_product = qfCreateProductFromDataConvention(
    value_date, "SOFR-1B-OVER-SOFR-1B-FLAT-ISZR", "10Y", 0.0
)
zs_engine = ValuationEngineProductRegistry().new_valuation_engine(
    yc_usd, zs_product, vpc, ValuationRequest.PV
)
zs_engine.calculate_value()
print("zero-spread PV:", float(zs_engine.value))

zero-spread PV: 0.0


#### Risk: curve-state gradient for each instrument, at its own par rate/spread

Each instrument's `get_risk()` runs its own `.backward()` and harvests `model.get_gradient(reset=True)`
-- one row per instrument, one column per curve-state node (`SOFR-1B`'s 10 tenors, then
`SOFR-1B-FLAT`'s single tenor). The zero-spread instrument's risk should land entirely on
`SOFR-1B-FLAT` and be ~0 on every `SOFR-1B` node: `DF(SOFR-1B-FLAT) = DF(SOFR-1B-FLAT own) *
DF(SOFR-1B)` (a `FundingIdentifier` chained on its reference), so
`DF_reference/DF_basis = DF(SOFR-1B-FLAT own)` -- the shared `SOFR-1B` factor cancels exactly in the
ratio, leaving no sensitivity to `SOFR-1B` at all.

In [ ]:
_ = yc_usd.get_gradient(reset=False)  # force gradient_lengths_/component_order_ to be populated
n_state = sum(yc_usd.gradient_lengths_)

labels = []
for key, length in zip(yc_usd.component_order_, yc_usd.gradient_lengths_):
    tenors_for_key = sofr_tenors if key == "SOFR-1B" else ["10Y"]
    labels.extend(f"{key}:{t}" for t in tenors_for_key[:length])

risk_rows = {}
for tenor, par_rate, engine in swap_engines:
    engine.calculate_value()
    grad = engine.grad_at_par()
    risk_rows[f"USD-SOFR-OIS {tenor}"] = grad

# grad = np.zeros(n_state)
zs_engine.calculate_value()
grad = zs_engine.grad_at_par()
risk_rows["SOFR-1B-OVER-SOFR-1B-FLAT-ISZR 10Y"] = grad

risk_df = pd.DataFrame(risk_rows, index=labels).T
risk_df

,SOFR-1B:1M,SOFR-1B:3M,SOFR-1B:6M,SOFR-1B:1Y,SOFR-1B:2Y,SOFR-1B:3Y,SOFR-1B:5Y,SOFR-1B:7Y,SOFR-1B:10Y,SOFR-1B:30Y,SOFR-1B-FLAT:10Y
USD-SOFR-OIS 1M,9238.780148,659.912868,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000e+00
USD-SOFR-OIS 3M,3034.697696,6502.923634,433.528242,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000e+00
USD-SOFR-OIS 6M,1541.691193,3303.623985,5175.677577,55.060400,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000e+00
USD-SOFR-OIS 1Y,788.511789,1689.668119,2647.146719,5040.843221,112.644541,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-5.987190e-14
USD-SOFR-OIS 2Y,401.254164,859.830352,1347.067551,2565.160549,5049.723645,41.295973,-0.000000,-0.000000,-0.000000,-0.000000,8.651309e+00
USD-SOFR-OIS 3Y,272.586297,584.113494,915.111141,1742.605258,3433.500320,3275.215030,36.096765,-0.000000,-0.000000,-0.000000,1.502365e+01
USD-SOFR-OIS 5Y,169.441888,363.089215,568.839769,1083.216157,2136.585633,2040.591854,3874.822281,26.052097,-0.000000,-0.000000,2.615042e+01
USD-SOFR-OIS 7Y,125.400683,268.715750,420.988008,801.668653,1581.772114,1511.094929,2870.981791,2671.823948,7.167201,-0.000000,2.831415e+01
USD-SOFR-OIS 10Y,92.476305,198.163231,310.455728,591.186972,1166.446846,1114.405095,2117.202955,1969.926045,2680.181812,11.765447,1.990862e+01
USD-SOFR-OIS 30Y,43.311444,92.810237,145.402704,276.883873,545.551594,520.401639,986.430043,914.925624,1239.086486,5399.794178,-8.169740e+01


In [ ]:
risk_df = pd.DataFrame(risk_rows, index=labels).T

risk_df.iloc[0, 1]

0.0

### Adding the Fed Funds Curve

Two new components, referencing `SOFR-1B` the same way `SOFR-1B-FLAT` does:
- `USD-Federal Funds-H.15-1B` -- overnight index curve, `REFERENCE: SOFR-1B`
- `USD-Federal Funds-H.15-1B-FLAT` -- flat funding overlay over it, `REFERENCE:
  USD-Federal Funds-H.15-1B`

(`fundingidentifiers.yaml` also has a `-CALIB` variant of each -- same mechanism, alternate naming
used elsewhere for a calibration-only proxy curve; not needed here, so `-FLAT` is used throughout,
consistent with `SOFR-1B-FLAT` above.)

**Chaining footgun worth calling out explicitly**: a `REFERENCE`d component's own
`INSTANTANEOUS FORWARD RATE` state is *not* the instrument's outright rate -- discount factors chain
multiplicatively (`DF(child) = DF(child's own state) * DF(reference)`), so the child's own IFR must
be just the *incremental* contribution on top of the reference. Feeding `SOFR-1B`'s own outright IFR
level into `USD-Federal Funds-H.15-1B`'s state (rather than a small basis spread) silently produces
a curve with roughly *double* the intended forward rate, not a realistic small Fed-Funds-over-SOFR
basis -- easy to get wrong by analogy with the state-data values used for `SOFR-1B` itself. Here the
Fed Funds curve's own state is a flat 3bp basis over `SOFR-1B` at every tenor.

The old `yc_usd`/`swap_engines`/`zs_engine` objects above stay valid after this rebuild -- each
engine holds a reference to the specific model instance it was built against, not to the `yc_usd`
name, so reassigning `yc_usd` to a new (4-component) model doesn't retroactively change what they
priced against.

In [ ]:
bm_list_v2 = list(bm_list)  # keep the original SOFR-1B / SOFR-1B-FLAT / USD entries
bm_list_v2.insert(
    2,
    qfCreateBuildMethod(
        "YC_OVERNIGHT_INDEX_ELEMENT",
        {
            "TARGET": "USD-Federal Funds-H.15-1B",
            "REFERENCE": "SOFR-1B",
            "INSTANTANEOUS FORWARD RATE": "USD-OIS-1B-IFR",
        },
    ),
)
bm_list_v2.insert(
    3,
    qfCreateBuildMethod(
        "YC_FUNDING_ELEMENT",
        {
            "TARGET": "USD-Federal Funds-H.15-1B-FLAT",
            "REFERENCE": "USD-Federal Funds-H.15-1B",
            "INSTANTANEOUS FORWARD RATE": "USD-OIS-1B-FLAT-IFR",
        },
    ),
)
build_method_collection_v2 = qfCreateModelBuildMethodCollection(bm_list_v2)

fedfunds_basis_ifr = [0.0003] * len(
    sofr_tenors
)  # flat 3bp basis over SOFR-1B, not an outright level

data_list_v2 = list(data_list)
df = pd.DataFrame(index=sofr_tenors)
df["values"] = fedfunds_basis_ifr
data_list_v2.append(qfCreateData1D(data_type, "USD-OIS-1B-IFR", df))

df = pd.DataFrame(index=["10Y"])
df["values"] = [0.0]
data_list_v2.append(qfCreateData1D(data_type, "USD-OIS-1B-FLAT-IFR", df))

data_collection_v2 = qfCreateDataCollection(data_list_v2)

yc_usd = qfCreateModel(value_date, "YIELD_CURVE", data_collection_v2, build_method_collection_v2)
print(
    f"USD yield curve rebuilt with {yc_usd.num_components} components:",
    yc_usd._gradient_component_order(),
)

print("SOFR-1B                        DF:", qfDiscountFactor(yc_usd, "SOFR-1B", expiry_date))
print(
    "USD-Federal Funds-H.15-1B       DF:",
    qfDiscountFactor(yc_usd, "USD-Federal Funds-H.15-1B", expiry_date),
)
print(
    "USD-Federal Funds-H.15-1B-FLAT  DF:",
    qfDiscountFactor(yc_usd, "USD-Federal Funds-H.15-1B-FLAT", expiry_date),
)

USD yield curve rebuilt with 4 components: ['SOFR-1B', 'SOFR-1B-FLAT', 'USD-FEDERAL FUNDS-H.15-1B', 'USD-FEDERAL FUNDS-H.15-1B-FLAT']
SOFR-1B                        DF: 0.9594847051399941
USD-Federal Funds-H.15-1B       DF: 0.9591969029009465
USD-Federal Funds-H.15-1B-FLAT  DF: 0.9591969029009465


### New Products Against the 4-Component Curve

Four more instrument types, each rebuilt at its own curve-implied par rate/spread the same two-pass
way as the `USD-SOFR-OIS` swaps above (throwaway build at `0.` -> read `par_rate_or_spread()` ->
rebuild at that value) via a shared `price_at_par` helper.

In [ ]:
def price_at_par(conv_name, axis1, label):
    guess = qfCreateProductFromDataConvention(value_date, conv_name, axis1, 0.0)
    guess_engine = ValuationEngineProductRegistry().new_valuation_engine(
        yc_usd, guess, vpc, ValuationRequest.PV
    )
    guess_engine.calculate_value()
    par = guess_engine.par_rate_or_spread()

    prod = qfCreateProductFromDataConvention(value_date, conv_name, axis1, par)
    engine = ValuationEngineProductRegistry().new_valuation_engine(
        yc_usd, prod, vpc, ValuationRequest.PV
    )
    engine.calculate_value()
    print(f"{label:40s} par={par: .6f}  PV={float(engine.value): .2e}")
    return engine, par

#### FRA / Fixing on `USD-Federal Funds-H.15-1B`

`USD-Federal Funds-H.15-1B-FRA` (`FRA OR FIXING`) dispatches to `ProductFactory.create_fra_or_fixing`
-> `ProductFRAOrFixing`. Since the index is overnight (`Federal Funds-H.15-1B`, native tenor `1D`),
`product.is_on` is `True` and `ValuationEngineProductFRAOrFixing` takes its ON branch (builds an
`AnchoredOvernightIndex`/`ValuationEngineAnalyticsCompositeIndex` pair by hand). `axis1='5Y'` makes
this a single 1-day fixing 5 years forward.

In [ ]:
fra_engine, fra_par = price_at_par("USD-Federal Funds-H.15-1B-FRA", "5Y", "FedFunds FRA/Fixing 5Y")

FedFunds FRA/Fixing 5Y                   par= 0.036298  PV= 0.00e+00


#### Overnight Index Future on `USD-Federal Funds-H.15-AVERAGE`

`FEDFUNDS-FUTURE` (`OVERNIGHT INDEX FUTURE`) projects off `USD-Federal Funds-H.15-AVERAGE`, an
`OVERNIGHT COMPOSITE INDEX` whose underlying `index` is exactly `USD-Federal Funds-H.15-1B` -- the
component just added above. Unlike the swaps/FRA, this data convention's `axis1` must be two
explicit dates (`"YYYY-MM-DD x YYYY-MM-DD"`), not a tenor.

In [ ]:
future_engine, future_par = price_at_par(
    "FEDFUNDS-FUTURE", "2026-09-16 x 2026-12-16", "FedFunds Future Sep26-Dec26"
)

FedFunds Future Sep26-Dec26              par= 0.041823  PV= 0.00e+00


#### Generic Forward Spread: `SOFR-1B-OVER-USD-Federal Funds-H.15-1B-SIMPLE-FWD-SPD`

`GENERIC FORWARD SPREAD` -> `ProductGenericForwardSpread`, priced by
`ValuationEngineProductGenericForwardSpread` -- spreads the implied forward rate of `SOFR-1B`
(basis) against `USD-Federal Funds-H.15-1B` (reference), `SIMPLE` compounding per the data
convention.

**Bug found and fixed while wiring this up**: the inherited `par_rate_or_spread()`/`pv01()` (from
`ValuationEngineProductGenericSpread`) assume the reference leg's own `pv01()` means
`d(PV)/d(the coupon that leg was built with)` -- true for a swap-style leg, but this engine's
reference leg is always a `ProductGenericForward` (payoff `notional*(F-K)*tau*df`), whose `pv01()`
means `d(PV)/d(F)` instead, the opposite sign relative to its own coupon `K`. Rebuilding the product
at the old (buggy) "par" spread doubled the PV in the wrong direction instead of zeroing it. Fixed by
flipping the sign of `spread_pv01_unit_` in a `calculate_value()` override specific to this
subclass (`fixedincomelib/yield_curve/valuation_engine.py`) -- the base class's own formula is left
untouched since it's independently correct for its (swap-based) use case.

In [ ]:
gfs_engine, gfs_par = price_at_par(
    "SOFR-1B-OVER-USD-Federal Funds-H.15-1B-SIMPLE-FWD-SPD",
    "5Y",
    "Generic Fwd Spread SOFR/FedFunds 5Y",
)

Generic Fwd Spread SOFR/FedFunds 5Y      par=-0.000357  PV= 0.00e+00


#### OIS Basis Swap: `USD-SOFR-COMPOUND-OVER-USD-Federal Funds-H.15-OIS-COMPOUND-BASIS-SWAP`

`OIS BASIS SWAP` -> `ProductOISBasisSwap`, priced by `ValuationEngineProductOISBasisSwap` -- both
legs are overnight composite indices (`basis overnight composite index: USD-SOFR-COMPOUND`,
`reference overnight composite index: USD-Federal Funds-H.15-COMPOUND`), spread quoted on the basis
(SOFR) leg.

**Bug found and fixed while wiring this up**: `ProductFactory.create_ois_basis_swap`
(`product/product_factory.py`) called `data_convention.basis_on_index.index.payment_holiday_conv()`
with parentheses -- `payment_holiday_conv` is a `@property`, not a method (the same gotcha
documented for `ProductFRAOrFixing` in the Bug Log), so this raised `TypeError: 'Calendar' object is
not callable` on every call. The same `()` typo turned out to be present in four more places across
`create_overnight_index_basis_swap`, `create_ois_basis_swap`, and
`create_cross_currency_basis_swap_non_mtm` -- all fixed (dropped the parentheses).

In [41]:
oisbs_engine, oisbs_par = price_at_par(
    "USD-SOFR-COMPOUND-OVER-USD-Federal Funds-H.15-OIS-COMPOUND-BASIS-SWAP",
    "5Y",
    "OIS Basis Swap SOFR/FedFunds 5Y",
)


oisbs_engine_1, oisbs_par_1 = price_at_par(
    "USD-SOFR-COMPOUND-OVER-USD-Federal Funds-H.15-OIS-COMPOUND-BASIS-SWAP",
    "7Y",
    "OIS Basis Swap SOFR/FedFunds 7Y",
)

oisbs_engine_2, oisbs_par_2 = price_at_par(
    "USD-SOFR-COMPOUND-OVER-USD-Federal Funds-H.15-OIS-COMPOUND-BASIS-SWAP",
    "10Y",
    "OIS Basis Swap SOFR/FedFunds 10Y",
)

OIS Basis Swap SOFR/FedFunds 5Y          par= 0.000299  PV= 2.33e-10
OIS Basis Swap SOFR/FedFunds 7Y          par= 0.000298  PV= 0.00e+00
OIS Basis Swap SOFR/FedFunds 10Y         par= 0.000298  PV= 1.86e-09


#### Risk for the four new instruments

Labels are read generically off each component's own `market_data` (tenor strings stored at index
build time) rather than hardcoded per-component tenor lists, since the curve now has two different
tenor grids (`SOFR-1B`/`USD-Federal Funds-H.15-1B` share the 10-tenor grid; both `-FLAT` components
have a single 10Y node).

The Generic Forward Spread's row is expected to be all-zero: `ValuationEngineProductGenericSpread`
combines its two legs' PVs as plain Python floats (`_to_float(...)` on each leg's `.value`), so the
wrapper's own `.value_` is never a live torch tensor -- there's no graph for the default
`get_risk()` to `.backward()` through. Curve risk isn't implemented for this engine family (noted
in `CLAUDE.md`); only PV/par-rate/pv01 reporting is supported today.

In [ ]:
_ = yc_usd.get_gradient(reset=False)
n_state_v2 = sum(yc_usd.gradient_lengths_)

labels_v2 = []
for key, length in zip(yc_usd.component_order_, yc_usd.gradient_lengths_):
    comp = yc_usd.retrieve_model_component(key)
    tenor_labels = [row[2] for row in comp.market_data]
    labels_v2.extend(f"{key}:{t}" for t in tenor_labels[:length])

new_risk_rows = {}
for name, eng in [
    ("FedFunds FRA/Fixing 5Y", fra_engine),
    ("FedFunds Future Sep26-Dec26", future_engine),
    ("Generic Fwd Spread SOFR/FedFunds 5Y", gfs_engine),
    ("OIS Basis Swap SOFR/FedFunds 5Y", oisbs_engine),
    ("OIS Basis Swap SOFR/FedFunds 7Y", oisbs_engine_1),
    ("OIS Basis Swap SOFR/FedFunds 10Y", oisbs_engine_2),
]:
    grad = np.zeros(n_state_v2)
    eng.calculate_value()
    eng.get_risk(gradient=grad)
    new_risk_rows[name] = grad

new_risk_df = pd.DataFrame(new_risk_rows, index=labels_v2).T
new_risk_df

,SOFR-1B:1M,SOFR-1B:3M,SOFR-1B:6M,SOFR-1B:1Y,SOFR-1B:2Y,SOFR-1B:3Y,SOFR-1B:5Y,SOFR-1B:7Y,SOFR-1B:10Y,SOFR-1B:30Y,...,USD-FEDERAL FUNDS-H.15-1B:3M,USD-FEDERAL FUNDS-H.15-1B:6M,USD-FEDERAL FUNDS-H.15-1B:1Y,USD-FEDERAL FUNDS-H.15-1B:2Y,USD-FEDERAL FUNDS-H.15-1B:3Y,USD-FEDERAL FUNDS-H.15-1B:5Y,USD-FEDERAL FUNDS-H.15-1B:7Y,USD-FEDERAL FUNDS-H.15-1B:10Y,USD-FEDERAL FUNDS-H.15-1B:30Y,USD-FEDERAL FUNDS-H.15-1B-FLAT:10Y
FedFunds FRA/Fixing 5Y,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2269.126298,0.000000,0.000000,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.269126e+03,0.000000e+00,0.000000,0.0
FedFunds Future Sep26-Dec26,-3.637979e-12,2032.778826,18521.504672,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,2.032779e+03,1.852150e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.0
Generic Fwd Spread SOFR/FedFunds 5Y,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.0
OIS Basis Swap SOFR/FedFunds 5Y,-5.590853e+01,-124.190280,-190.893192,-358.442875,-717.563858,-683.795728,-1289.734840,-8.556413,0.000000,0.000000,...,-1.642591e+06,-2.546215e+06,-4.774578e+06,-9.494789e+06,-9.070263e+06,-1.725215e+07,-1.144199e+05,0.000000e+00,0.000000,0.0
OIS Basis Swap SOFR/FedFunds 7Y,-5.796802e+01,-124.217179,-191.005678,-358.775518,-718.880446,-681.492599,-1302.107791,-1209.695048,-3.214107,0.000000,...,-1.642789e+06,-2.547004e+06,-4.774574e+06,-9.495984e+06,-9.068369e+06,-1.725179e+07,-1.608902e+07,-4.255925e+04,0.000000,0.0
OIS Basis Swap SOFR/FedFunds 10Y,-5.590864e+01,-124.190545,-190.928532,-358.615797,-718.336945,-685.106490,-1293.919142,-1208.192110,-1645.763231,-7.092253,...,-1.642591e+06,-2.546215e+06,-4.774578e+06,-9.494789e+06,-9.070263e+06,-1.725215e+07,-1.608784e+07,-2.191257e+07,-94812.525244,0.0


### Full Jacobian: All 16 Instruments Against the Current (4-Component) Curve

One combined matrix, `J[i, j] = d(PV_i)/d(state_j)`, across every instrument built in this notebook:
the 10 `USD-SOFR-OIS` swaps, the `SOFR-1B-OVER-SOFR-1B-FLAT-ISZR` zero-spread instrument, the Fed
Funds FRA/Fixing, the Fed Funds Future, the Generic Forward Spread, and the OIS Basis Swap -- 16 rows
x 22 columns (the same `risk_df`/`new_risk_df` idea from above, just consolidated into one table).

This is a **PV-risk Jacobian** (autograd `get_risk()` on each instrument's PV, at its own par
rate/spread), not a **calibration Jacobian** (`d(par_rate)/d(state)`, which is what `grad_at_par()` +
the commented-out `YieldCurve.calculate_model_jacobian()` would build, per the discussion above --
still not implemented anywhere in this codebase). The two are easy to conflate: a PV-risk Jacobian
answers "how much does this instrument's mark move if the curve wiggles", a calibration Jacobian
answers "how much does the *par quote* implied by the curve move" -- the latter is what a
bootstrap/Newton solver would need, not what's built here.

The 10 swaps and the zero-spread instrument have to be **rebuilt** against the current `yc_usd`
first: `swap_engines`/`zs_engine` above were priced against the *original* 2-component curve (before
the Fed Funds curve existed), so their own `get_risk()` only ever returns an 11-long vector -- too
short to sit in the same 22-column matrix as the Fed-Funds-dependent instruments. Rebuilding them
here (via the same `price_at_par` helper) is also a useful check in its own right: they still price
to par on the augmented curve, confirming that adding `USD-Federal Funds-H.15-1B`/`-FLAT` (which only
*reference* `SOFR-1B`, never feed back into it) left `SOFR-1B`'s own calibration undisturbed.

In [ ]:
all_engines = {}

for tenor in sofr_tenors:
    eng, _ = price_at_par("USD-SOFR-OIS", tenor, f"USD-SOFR-OIS {tenor}")
    all_engines[f"USD-SOFR-OIS {tenor}"] = eng

zs_engine_v2, _ = price_at_par(
    "SOFR-1B-OVER-SOFR-1B-FLAT-ISZR", "10Y", "SOFR-1B-OVER-SOFR-1B-FLAT-ISZR 10Y"
)
all_engines["SOFR-1B-OVER-SOFR-1B-FLAT-ISZR 10Y"] = zs_engine_v2

all_engines["FedFunds FRA/Fixing 5Y"] = fra_engine
all_engines["FedFunds Future Sep26-Dec26"] = future_engine
all_engines["Generic Fwd Spread SOFR/FedFunds 5Y"] = gfs_engine
all_engines["OIS Basis Swap SOFR/FedFunds 5Y"] = oisbs_engine
all_engines["OIS Basis Swap SOFR/FedFunds 7Y"] = oisbs_engine_1
all_engines["OIS Basis Swap SOFR/FedFunds 10Y"] = oisbs_engine_2

USD-SOFR-OIS 1M                          par= 0.043410  PV= 0.00e+00
USD-SOFR-OIS 3M                          par= 0.042902  PV= 0.00e+00
USD-SOFR-OIS 6M                          par= 0.042491  PV= 0.00e+00
USD-SOFR-OIS 1Y                          par= 0.041580  PV=-5.82e-11
USD-SOFR-OIS 2Y                          par= 0.039892  PV= 0.00e+00
USD-SOFR-OIS 3Y                          par= 0.039000  PV= 0.00e+00
USD-SOFR-OIS 5Y                          par= 0.037924  PV= 2.33e-10
USD-SOFR-OIS 7Y                          par= 0.037585  PV= 4.66e-10
USD-SOFR-OIS 10Y                         par= 0.037611  PV= 4.66e-10
USD-SOFR-OIS 30Y                         par= 0.039000  PV=-3.73e-09
SOFR-1B-OVER-SOFR-1B-FLAT-ISZR 10Y       par=-0.000000  PV= 0.00e+00


In [ ]:
_ = yc_usd.get_gradient(reset=False)
n_state_all = sum(yc_usd.gradient_lengths_)

labels_all = []
for key, length in zip(yc_usd.component_order_, yc_usd.gradient_lengths_):
    comp = yc_usd.retrieve_model_component(key)
    tenor_labels = [row[2] for row in comp.market_data]
    labels_all.extend(f"{key}:{t}" for t in tenor_labels[:length])

jacobian_rows = {}
for name, eng in all_engines.items():
    grad = np.zeros(n_state_all)
    eng.calculate_value()
    eng.get_risk(gradient=grad)
    jacobian_rows[name] = grad

jacobian_df = pd.DataFrame(jacobian_rows, index=labels_all).T
print(f"Jacobian shape: {jacobian_df.shape[0]} instruments x {jacobian_df.shape[1]} state nodes")
jacobian_df

Jacobian shape: 17 instruments x 22 state nodes


,SOFR-1B:1M,SOFR-1B:3M,SOFR-1B:6M,SOFR-1B:1Y,SOFR-1B:2Y,SOFR-1B:3Y,SOFR-1B:5Y,SOFR-1B:7Y,SOFR-1B:10Y,SOFR-1B:30Y,...,USD-FEDERAL FUNDS-H.15-1B:3M,USD-FEDERAL FUNDS-H.15-1B:6M,USD-FEDERAL FUNDS-H.15-1B:1Y,USD-FEDERAL FUNDS-H.15-1B:2Y,USD-FEDERAL FUNDS-H.15-1B:3Y,USD-FEDERAL FUNDS-H.15-1B:5Y,USD-FEDERAL FUNDS-H.15-1B:7Y,USD-FEDERAL FUNDS-H.15-1B:10Y,USD-FEDERAL FUNDS-H.15-1B:30Y,USD-FEDERAL FUNDS-H.15-1B-FLAT:10Y
USD-SOFR-OIS 1M,-7.665728e+05,-5.475520e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.0
USD-SOFR-OIS 3M,-7.665770e+05,-1.642665e+06,-1.095110e+05,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.0
USD-SOFR-OIS 6M,-7.665854e+05,-1.642683e+06,-2.573537e+06,-2.737805e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.0
USD-SOFR-OIS 1Y,-7.665938e+05,-1.642701e+06,-2.573565e+06,-4.900725e+06,-1.095134e+05,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.0
USD-SOFR-OIS 2Y,-7.665938e+05,-1.642701e+06,-2.573565e+06,-4.900725e+06,-9.647469e+06,-7.889573e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.0
USD-SOFR-OIS 3Y,-7.665938e+05,-1.642701e+06,-2.573565e+06,-4.900725e+06,-9.656025e+06,-9.210880e+06,-1.015149e+05,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.0
USD-SOFR-OIS 5Y,-7.665142e+05,-1.642528e+06,-2.573294e+06,-4.900209e+06,-9.665398e+06,-9.231146e+06,-1.752876e+07,-1.178534e+05,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.0
USD-SOFR-OIS 7Y,-7.665938e+05,-1.642701e+06,-2.573565e+06,-4.900725e+06,-9.669618e+06,-9.237558e+06,-1.755076e+07,-1.633327e+07,-4.381421e+04,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.0
USD-SOFR-OIS 10Y,-7.665148e+05,-1.642529e+06,-2.573296e+06,-4.900213e+06,-9.668410e+06,-9.237048e+06,-1.754901e+07,-1.632826e+07,-2.221541e+07,-9.752108e+04,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.0
USD-SOFR-OIS 30Y,-7.665938e+05,-1.642701e+06,-2.573565e+06,-4.900725e+06,-9.656027e+06,-9.210884e+06,-1.745939e+07,-1.619379e+07,-2.193130e+07,-9.557402e+07,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.0


In [46]:
jacobian_df.to_clipboard(excel=True)